In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [54]:
from google.colab import userdata
MODEL = "nvidia/nemotron-3.5-lightning:free"
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')


In [6]:
openai_api_key = userdata.get('OPENROUTER_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

DB = "prices.db"

OpenAI API Key exists and begins sk-or-v1


In [7]:
model = MODEL
openrouter = OpenAI(api_key = openai_api_key , base_url = 'https://openrouter.ai/api/v1')

In [8]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [9]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [10]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


OperationalError: no such table: prices

In [29]:
TTS_MODEL = "fish-audio/s2.1-pro-free:free"

In [30]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [31]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1918793de67d191dd6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
def chat(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openrouter.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

        response = openrouter.chat.completions.create(
            model=MODEL,
            reasoning_effort="none",
            messages=messages,
            tools=tools
        )

    # Extract Nemotron's generated text
    reply = response.choices[0].message.content

    # Convert text into audio
    audio_bytes = talker(reply)

    # Save MP3
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as audio_file:
        audio_file.write(audio_bytes)
        audio_path = audio_file.name

    return reply, audio_path

In [37]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",

                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [41]:
MODEL = "nvidia/nemotron-3.5-lightning:free"

TTS_MODEL = "fish-audio/s2.1-pro-free:free"
TTS_VOICE = "default"

In [48]:
import requests

def talker(message):
    if not isinstance(message, str) or not message.strip():
        raise ValueError("Text for speech generation cannot be empty.")

    url = "https://openrouter.ai/api/v1/audio/speech"

    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
        "Accept": "application/octet-stream"
    }

    payload = {
        "model": "fish-audio/s2.1-pro-free:free",
        "input": message.strip()
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=120
    )

    if not response.ok:
        raise RuntimeError(
            f"Fish Audio request failed ({response.status_code}): "
            f"{response.text}"
        )

    audio_bytes = response.content

    if not audio_bytes:
        raise RuntimeError("Fish Audio returned an empty response.")

    return audio_bytes

In [45]:
import requests

url = "https://openrouter.ai/api/v1/models"

response = requests.get(
    url,
    headers={"Authorization": f"Bearer {openai_api_key}"},
    params={"output_modalities": "speech"},
    timeout=30
)

response.raise_for_status()
models_data = response.json()["data"]

fish_models = [
    model for model in models_data
    if "fish-audio" in model.get("id", "").lower()
]

for model in fish_models:
    print("\nModel ID:", model.get("id"))
    print("Name:", model.get("name"))
    print("Supported voices:", model.get("supported_voices"))


Model ID: fish-audio/drama-3
Name: Fish Audio: Drama 3
Supported voices: None

Model ID: fish-audio/s1
Name: Fish Audio: S1
Supported voices: None

Model ID: fish-audio/s2-pro
Name: Fish Audio: S2 Pro
Supported voices: None

Model ID: fish-audio/s2.1-pro-free:free
Name: Fish Audio: S2.1 Pro Free (free)
Supported voices: None

Model ID: fish-audio/s2.1-pro
Name: Fish Audio: S2.1 Pro
Supported voices: None


In [39]:
def chat(message, history):
    """FlightAI chatbot with Nemotron, SQLite tools and Fish Audio."""

    if not message or not message.strip():
        return "Please enter a message.", None

    # Build conversation messages
    messages = [
        {"role": "system", "content": system_message}
    ]

    # Add previous conversation history
    for item in history or []:
        role = item.get("role")
        content = item.get("content")

        if role in ("user", "assistant") and isinstance(content, str):
            messages.append({
                "role": role,
                "content": content
            })

    # Add the latest user message
    messages.append({
        "role": "user",
        "content": message.strip()
    })

    try:
        # STEP 1: Call Nemotron
        logger.info("Calling Nemotron...")

        response = openrouter.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )

        # STEP 2: Handle tool calls
        tool_iterations = 0
        max_tool_iterations = 5

        while response.choices[0].finish_reason == "tool_calls":

            if tool_iterations >= max_tool_iterations:
                raise RuntimeError("Maximum tool-call limit reached.")

            assistant_message = response.choices[0].message

            # Preserve the assistant's tool-call message in the conversation
            messages.append(
                assistant_message.model_dump(exclude_none=True)
            )

            # Execute your existing SQLite tool handler
            tool_responses = handle_tool_calls(assistant_message)

            if not tool_responses:
                raise RuntimeError("Tool handler returned no responses.")

            messages.extend(tool_responses)

            tool_iterations += 1

            logger.info("Executing tool call iteration %s", tool_iterations)

            # Ask Nemotron to process the tool results
            response = openrouter.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                reasoning_effort="none"
            )

        # STEP 3: Extract final text
        reply = response.choices[0].message.content

        if not reply or not reply.strip():
            raise RuntimeError("Nemotron returned an empty response.")

        reply = reply.strip()

        logger.info("Nemotron response generated successfully.")

    except Exception as e:
        logger.exception("Nemotron or tool execution failed.")
        return f"Sorry, I encountered an error: {str(e)}", None

    # STEP 4: Generate audio using Fish Audio
    try:
        logger.info("Generating speech with Fish Audio...")

        audio_bytes = talker(reply)

        # STEP 5: Save MP3
        with tempfile.NamedTemporaryFile(
            mode="wb",
            suffix=".mp3",
            delete=False
        ) as audio_file:

            audio_file.write(audio_bytes)
            audio_path = audio_file.name

        logger.info("Audio generated successfully.")

        return reply, audio_path

    except Exception as e:
        logger.exception("Fish Audio generation failed.")

        # Keep the chatbot usable even if TTS fails
        return reply + "\n\n(Audio generation failed. Please check the notebook logs.)", None

In [49]:
import tempfile
from IPython.display import Audio, display

test_text = (
    "Hello! Welcome to FlightAI. "
    "I can help you find flights and plan your next journey."
)

print("Testing Fish Audio through OpenRouter...")

try:
    audio_bytes = talker(test_text)

    with tempfile.NamedTemporaryFile(
        suffix=".mp3",
        delete=False
    ) as audio_file:
        audio_file.write(audio_bytes)
        audio_path = audio_file.name

    print("SUCCESS: Audio generated.")
    print("Audio size:", len(audio_bytes), "bytes")

    display(Audio(filename=audio_path))

except Exception as e:
    print("FAILED:", type(e).__name__, str(e))
    raise

Testing Fish Audio through OpenRouter...
SUCCESS: Audio generated.
Audio size: 495616 bytes


In [50]:
content_type = response.headers.get("Content-Type", "").lower()

if "mpeg" in content_type or "mp3" in content_type:
    extension = ".mp3"
elif "wav" in content_type:
    extension = ".wav"
elif "ogg" in content_type:
    extension = ".ogg"
elif "flac" in content_type:
    extension = ".flac"
else:
    extension = ".audio"

with tempfile.NamedTemporaryFile(
    suffix=extension,
    delete=False
) as audio_file:
    audio_file.write(response.content)
    audio_path = audio_file.name

print("Saved file:", audio_path)
print("Detected extension:", extension)

display(Audio(filename=audio_path, autoplay=False))

Saved file: /tmp/tmpz54wxrng.audio
Detected extension: .audio


In [51]:
content_type = response.headers.get("Content-Type", "").lower()

if "mpeg" in content_type or "mp3" in content_type:
    extension = ".mp3"
elif "wav" in content_type:
    extension = ".wav"
elif "ogg" in content_type:
    extension = ".ogg"
elif "flac" in content_type:
    extension = ".flac"
else:
    extension = ".audio"

with tempfile.NamedTemporaryFile(
    suffix=extension,
    delete=False
) as audio_file:
    audio_file.write(response.content)
    audio_path = audio_file.name

print("Saved file:", audio_path)
print("Detected extension:", extension)

display(Audio(filename=audio_path, autoplay=False))

Saved file: /tmp/tmpbsjwck88.audio
Detected extension: .audio


In [52]:
print("Status:", response.status_code)

print("\nResponse headers:")
for key, value in response.headers.items():
    print(f"{key}: {value}")

print("\nFirst 32 bytes:")
print(response.content[:32])

print("\nFirst 32 bytes (hex):")
print(response.content[:32].hex())

print("\nResponse size:", len(response.content), "bytes")

print("\nFirst 300 bytes as text (if readable):")
print(response.content[:300].decode("utf-8", errors="replace"))

Status: 200

Response headers:
Date: Mon, 02 Nov 2026 17:29:43 GMT
Content-Type: application/json
Transfer-Encoding: chunked
Connection: keep-alive
CF-Cache-Status: MISS
Access-Control-Allow-Origin: *
Cache-Control: private, max-age=5, stale-while-revalidate=5
Vary: Host, Authorization, X-Api-Key
Access-Control-Expose-Headers: X-Generation-Id,X-Provider-Name,request-id,cf-ray
set-cookie: __cf_bm=aldA.aStx_zz9L4L07voejQ.33NOpw2sl14J.baQZTE-1793640583.2760136-1.0.1.1-upTDaQclRzReHgvQWWeX_1UxHYtCdUXFHYvrmybhPJLmLR_edf7MfQyDGJ_KOByeXo_22wGns7PHfalxK.SI4svKh32AKTh8iYwTf24UvqgcIUvx2snrG7GhG7elToaq; HttpOnly; SameSite=None; Secure; Path=/; Domain=openrouter.ai; Expires=Mon, 02 Nov 2026 17:59:43 GMT
Permissions-Policy: payment=(self "https://checkout.stripe.com" "https://connect-js.stripe.com" "https://js.stripe.com" "https://*.js.stripe.com" "https://hooks.stripe.com")
Referrer-Policy: no-referrer, strict-origin-when-cross-origin
X-Content-Type-Options: nosniff
Content-Encoding: gzip
Server: 

In [53]:
data = response.content

if data.startswith(b"ID3"):
    print("Detected: MP3 (ID3 header)")
elif data.startswith(b"RIFF"):
    print("Detected: WAV")
elif data.startswith(b"OggS"):
    print("Detected: OGG")
elif data.startswith(b"fLaC"):
    print("Detected: FLAC")
elif data.startswith(b"\xff\xfb") or data.startswith(b"\xff\xf3") or data.startswith(b"\xff\xf2"):
    print("Possible MP3 frame")
else:
    print("Unknown audio signature")

Unknown audio signature


In [55]:
import requests
import json

tts_url = "https://openrouter.ai/api/v1/audio/speech"

tts_headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}

tts_payload = {
    "model": "fish-audio/s2.1-pro-free:free",
    "input": "Hello! Welcome to FlightAI. I can help you find flights."
}

tts_response = requests.post(
    tts_url,
    headers=tts_headers,
    json=tts_payload,
    timeout=120
)

print("Status:", tts_response.status_code)
print("Content-Type:", tts_response.headers.get("Content-Type"))
print("Response size:", len(tts_response.content))
print("First 32 bytes:", tts_response.content[:32])

Status: 200
Content-Type: audio/pcm;rate=44100;channels=1
Response size: 397312
First 32 bytes: b'\xaf\xff\x94\xff\x8e\xffo\xff`\xffj\xff\xa1\xff\xd6\xff\xf3\xff\x16\x00A\x00y\x00\x96\x00~\x00Y\x00W\x00'


In [56]:
if "application/json" in tts_response.headers.get("Content-Type", ""):
    print("JSON response received:")
    print(json.dumps(tts_response.json(), indent=2)[:3000])
else:
    print("Non-JSON response received.")
    print("Audio signature:", tts_response.content[:16])

Non-JSON response received.
Audio signature: b'\xaf\xff\x94\xff\x8e\xffo\xff`\xffj\xff\xa1\xff\xd6\xff'


In [57]:
import wave
import tempfile
from IPython.display import Audio, display

# Fish Audio response contains raw PCM audio
pcm_bytes = tts_response.content

# Convert raw PCM into a WAV file
with tempfile.NamedTemporaryFile(
    suffix=".wav",
    delete=False
) as audio_file:

    audio_path = audio_file.name

with wave.open(audio_path, "wb") as wav_file:
    wav_file.setnchannels(1)       # Mono
    wav_file.setsampwidth(2)       # 16-bit PCM
    wav_file.setframerate(44100)   # 44.1 kHz
    wav_file.writeframes(pcm_bytes)

print("WAV file created successfully!")
print("File:", audio_path)

display(Audio(filename=audio_path, autoplay=False))

WAV file created successfully!
File: /tmp/tmpx4palv_v.wav


In [58]:
import requests
import wave
import tempfile

def talker(message):

    url = "https://openrouter.ai/api/v1/audio/speech"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": TTS_MODEL,
        "input": message.strip()
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    pcm_bytes = response.content

    with tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ) as audio_file:

        audio_path = audio_file.name

    with wave.open(audio_path, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(44100)
        wav_file.writeframes(pcm_bytes)

    return audio_path

In [59]:
import requests
import wave
import tempfile

def talker(message):

    if not message or not message.strip():
        return None

    url = "https://openrouter.ai/api/v1/audio/speech"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": TTS_MODEL,
        "input": message.strip()
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    pcm_bytes = response.content

    with tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ) as audio_file:
        audio_path = audio_file.name

    with wave.open(audio_path, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(44100)
        wav_file.writeframes(pcm_bytes)

    return audio_path

In [60]:
def chat(message, history):

    messages = [
        {"role": "system", "content": system_message}
    ]

    messages.extend(history)

    messages.append({
        "role": "user",
        "content": message
    })

    response = openrouter.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        assistant_message = response.choices[0].message

        tool_responses = handle_tool_calls(assistant_message)

        messages.append(assistant_message)
        messages.extend(tool_responses)

        response = openrouter.chat.completions.create(
            model=MODEL,
            reasoning_effort="none",
            messages=messages,
            tools=tools
        )

    # Final response generated by Nemotron
    reply = response.choices[0].message.content

    # Convert Nemotron's final response into speech
    audio_path = talker(reply)

    return reply, audio_path

In [62]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat,
    additional_outputs=[
        gr.Audio(
            label="FlightAI Voice Response",
            type="filepath",
            autoplay=True
        )
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://442673c86cfc5acb9b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [64]:
import gradio as gr

with gr.Blocks() as demo:

    chatbot = gr.ChatInterface(
        fn=chat,
        additional_outputs=[
            gr.Audio(
                label="FlightAI Voice Response",
                type="filepath",
                autoplay=True
            )
        ]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0566d6deef26e1c4d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
